# N134 · 坐标压缩与秩

**目标：** 保持相对次序而不误保留原始距离。

**先修：** N037, N042。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 去重排序；值到秩；离线预处理；相等键；下界查询；秩与差值区别。

**配套讲解来源：** [同名逐章意见](../../comment/134_coordinate_compression_order_statistics.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决的是"值的范围太大，但真正有用的只有大小关系"的问题。很多题目给你的坐标可能是 `-7`、`20`、甚至 `10^12` 这种天文数字，但你后续想用的数据结构（比如树状数组）只能开一个普通的 Python 列表，列表下标不可能开到 `10^12` 那么大。坐标压缩（coordinate compression）就是把这一堆稀稀拉拉的大数字，重新编号成 `0, 1, 2, ...` 这样连续的小整数，同时保证：原来大的数编号仍然大，原来小的数编号仍然小，原来相等的数编号相同。

一个具体到数字的例子（就是 “运行示例”部分 用的那个）：输入 `[10^12, -7, 20, -7]`，输出零基秩 `[2, 0, 1, 0]`。为什么是它？我们把所有出现过的值去重再排序，得到有序表 `[-7, 20, 10^12]`，它们的位置分别是 0、1、2；于是 `-7 → 0`、`20 → 1`、`10^12 → 2`，再按输入原顺序替换回去就是 `[2, 0, 1, 0]`。注意两个 `-7` 出现了两次，它们共享同一个秩 0；`10^12` 虽然大得离谱，也只分到秩 2。相对次序完整保留，但数值距离被彻底丢掉了：原值最大最小相差 `10^12 - (-7) = 1000000000007`，而秩最大最小只相差 2——“运行示例”部分 最后打印的 `秩不能保持差值： 1000000000007 ≠ 2` 说的就是这件事。

本章还顺带解决第二个问题：给定数组，求每个位置"右边有多少个数比它小"。例如输入 `[5, 2, 6, 1]` → 输出 `[2, 1, 1, 0]`，因为 5 的右边有 2 和 1 两个更小的数，2 的右边只有 1 一个更小的数，6 的右边只有 1，而 1 的右边没有数。这题是坐标压缩思想（用有序结构做"排名查询"）的直接应用。

为什么"巨大坐标开不了数组"是实际问题？举个规模：值域到 10^12 时，就算每个整数只占一个 Python 整数引用 8 字节，开满也要 8000 GB 内存；而压缩后只需要 n 个格子。也就是说，压缩是把"值域大小"的问题转换成"元素个数"的问题，这一步转换在后面几章（树状数组、线段树）会反复出现。

## 2. 基础知识：先读懂这些词

- **坐标压缩（coordinate compression）**：把任意一堆值（可能巨大、稀疏、有负数）映射成从 0 开始的连续整数编号。你需要它是因为数组下标不能是 `10^12`，但可以是 3。
- **秩（rank）**：一个值在所有出现过的值里"排第几"。零基秩从 0 开始数，一基秩从 1 开始数。`rank_transform` 返回的就是一基秩。
- **去重排序**：先用 `set` 把重复值合并，再 `sorted` 排成严格递增的表。为什么去重？因为两个相等的值必须拿到同一个秩，否则相等关系就被破坏了。
- **离线预处理**：意思是"先把所有输入都读进来，整体处理一遍，再回答问题"。坐标压缩必须离线做，因为你不知道全部值就没法排全集的名次。
- **秩与差值的区别**：压缩只保留"谁大谁小、谁等于谁"，不保留"大多少"。原值 `10^12 - (-7) = 1000000000007`，压缩后两个秩的差只有 `3 - 0 = 3`，这两个差完全不是一回事。
- **下界查询（bisect_left）**：`bisect_left(ordered, x)` 返回 x 在有序列表中"应该插进去且不打乱顺序的最左位置"，这个数字恰好等于列表里严格小于 x 的元素个数。它和 `bisect_right` 的区别就在等值元素上：列表里有 3 个 5 时，查 5 用 left 得"5 之前的个数"，用 right 得"含 5 在内的个数"——本章要"严格更小"，所以选 left。
- **右侧更小数（count of smaller numbers after self）**：对每个位置统计它右边比自己小的元素个数，这是许多"逆序类"题目的基础形态。

## 3. 思路推导：从小例子开始

我们先从最简单的"给每个数重新编号"推导起，再推导"右侧更小数"。

- **Step 1**：把输入里所有值收进 `set` 去重。这一步保证相等的值只会被编号一次，它们天然共享同一个秩。
- **Step 2**：把去重后的集合排序，得到严格递增的有序表 `ordered`。这一步确立"值 ↔ 名次"的对应关系：排得越靠前，名次越小。
- **Step 3**：用字典 `rank` 记录"值 → 它在 ordered 里的下标"。查一次字典是 O(1)，比每次都去 ordered 里二分更快也更清晰。
- **Step 4**：按输入原来的顺序，把每个值换成它的秩，输出秩数组。到这一步，坐标压缩就完成了。
- **Step 5**（rank_transform）：在零基秩上每个元素加 1，得到一基秩。加 1 是因为有些题目（如 LeetCode 1331）规定秩从 1 开始。
- **Step 6**（count_smaller_sorted_list）：从右往左扫数组，维护一个"已经见过的右侧元素"的有序列表。对当前元素 x，`bisect_left(ordered, x)` 给出列表里严格小于 x 的个数，这正是"x 右侧比 x 小的个数"；然后把 x 用 `insort` 插进列表，继续处理下一个。

**手算演示（coordinate_compress 本身，输入 `[7, 7, -2]`，来自 “自动测试”部分 的第二条测试）**：

| 步骤 | 做什么 | 中间结果 |
|---|---|---|
| 1 | 收进 set 去重 | `{7, -2}`（两个 7 只剩一个） |
| 2 | sorted 排序 | `[-2, 7]`（-2 更小，排前面） |
| 3 | 建字典"值→下标" | `{-2: 0, 7: 1}` |
| 4 | 按输入顺序换秩 | 输入第 1 个是 7 → 1；第 2 个是 7 → 1；第 3 个是 -2 → 0 |
| 5 | 返回 | `ordered=[-2,7]`，零基秩 `[1, 1, 0]` |

再套 `rank_transform` 每个加 1，就得到测试断言里的 `[2, 2, 1]`。这个演示里要特别注意第 1 步和第 4 步：去重决定"相等的值共享秩"，按原顺序换秩决定"输出和输入一一对应"，这两点合起来就是"保序"的全部含义。

**手算演示（count_smaller_sorted_list，输入 `[5, 2, 6, 1]`）**：我们从右往左扫。

| 步骤 | 当前元素 x | 此时有序列表（插入前） | bisect_left 结果（右侧比 x 小的个数） | 插入后的列表 |
|---|---|---|---|---|
| 1 | 1 | `[]` | 0（列表空，没有比 1 小的） | `[1]` |
| 2 | 6 | `[1]` | 1（只有 1 比 6 小） | `[1, 6]` |
| 3 | 2 | `[1, 6]` | 1（只有 1 比 2 小，6 不算） | `[1, 2, 6]` |
| 4 | 5 | `[1, 2, 6]` | 2（1 和 2 都比 5 小） | `[1, 2, 5, 6]` |

我们收集到的答案是反着的：`[0, 1, 1, 2]`（按 1、6、2、5 的处理顺序），最后一步 `answer[::-1]` 把它翻回来，得到 `[2, 1, 1, 0]`，正好和第一节例子的期望输出一致。你也可以换个角度自查：答案数组的最后一个元素永远是 0，因为最右边的元素右边没有任何数。

这个倒序扫描还有两个不变量值得口头复述一遍：

- 处理位置 i 之前，有序列表里装的都是下标大于 i 的元素，一个不多（左边的还没走到）、一个不少（右侧的全部已插入）。
- 查询永远发生在插入自己之前，所以列表里不可能有 x 自己，也就不会出现"把自己算成比自己小"的低级错误。

再看 “运行示例”部分 的小实例：`a = [10^12, -7, 20, -7]`，代码用 `show_table` 打出"原值 / 零基秩"对照表（四行分别是 `[10^12, 2]`、`[-7, 0]`、`[20, 1]`、`[-7, 0]`），并额外打印一行 `秩不能保持差值： 1000000000007 ≠ 2`——意思是原值最大最小相差一万亿多，而秩最大最小只相差 2，这就是"压缩保留次序、丢掉距离"的活例子。

## 4. 核心代码：coordinate_compress

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def coordinate_compress(values):
    ordered = sorted(set(values))
    rank = {v: i for i, v in enumerate(ordered)}
    return (ordered, [rank[v] for v in values])
```

### 逐段读懂代码

“分段实现”部分 只有三段代码，我们逐个看。

**第一个函数：coordinate_compress(values)**

```python
from bisect import bisect_left, insort

def coordinate_compress(values):
    ordered = sorted(set(values))
    rank = {v: i for i, v in enumerate(ordered)}
    return (ordered, [rank[v] for v in values])
```

第一行从标准库 `bisect` 导入两个工具：`bisect_left` 是二分查找左边界，`insort` 是"二分找位置 + 插入"一条龙。

函数体第一句 `ordered=sorted(set(values))` 一句干两件事：`set(values)` 去重，`sorted(...)` 排序，得到严格递增的有序表。第二句用字典推导式 `{v:i for i,v in enumerate(ordered)}` 建"值 → 下标"的查询表，比如 `ordered=[-7,20,10^12]`，那么 `rank` 就是 `{-7:0, 20:1, 10^12:2}`。最后 `return ordered,[rank[v] for v in values]` 同时返回两样东西：有序表本身，以及把输入逐个换成秩之后的新列表。注意列表推导是按 `values` 原顺序走的，所以秩数组和输入数组一一对应。

**第二个函数：rank_transform(arr)**

```python
def rank_transform(arr):
    ordered, ranks = coordinate_compress(arr)
    return [r + 1 for r in ranks]
```

它直接复用上一个函数：先拿到零基秩，然后每个元素加 1 变成一基秩。为什么单独封装？因为"从 1 开始的秩"是 LeetCode 1331 这类题目的要求，封装成函数可以和零基版本区分开，用的时候不会搞混。拿测试用例过一遍：输入 `[40,10,20,30]`，零基秩是 `[3,0,1,2]`，加 1 得 `[4,1,2,3]`，正是 “自动测试”部分 第一条断言的期望值；复用而不是重写一遍排序逻辑，也保证了两个函数永远不会"行为不一致"。

**第三个函数：count_smaller_sorted_list(nums)**

```python
def count_smaller_sorted_list(nums):
    ordered = []
    answer = []
    for x in reversed(nums):
        answer.append(bisect_left(ordered, x))
        insort(ordered, x)
    return answer[::-1]
```

`ordered` 是"右侧已见元素"的有序列表，一开始是空的；`answer` 收集每个位置的结果。`reversed(nums)` 让我们从最后一个元素往前走。循环体一行干两件事：先 `bisect_left(ordered,x)` 问"列表里有几个数严格小于 x"，把答案追加进 `answer`；再用 `insort(ordered,x)` 把 x 插进正确位置，让列表保持有序。注意这两步的顺序不能颠倒——必须先查询再插入，否则 x 会把自己也算进去。最后 `answer[::-1]` 反转列表，因为我们是倒着处理的，收集的答案顺序也是倒的，翻回来才和输入下标对齐。

顺带讲一下 “运行示例”部分 的展示代码（它不属于 “分段实现”部分 的核心实现，但帮你直观看到压缩结果）：

```python
a = [10 ** 12, -7, 20, -7]

ordered, ranks = coordinate_compress(a)

show_table(['原值', '零基秩'], list(zip(a, ranks)))

print('秩不能保持差值：', ordered[-1] - ordered[0], '≠', len(ordered) - 1)
```

第一行调用 `coordinate_compress` 拿到有序表和秩数组；第二行 `list(zip(a,ranks))` 把原值和秩按位置配成四对交给 `show_table` 渲染；第三行故意打印两个不相等的数：`ordered[-1]-ordered[0]` 是原值最大最小之差（1000000000007），`len(ordered)-1` 是秩的最大可能差（2）。一行 print 就把"压缩丢距离"演示得明明白白。

## 5. 分段实现：按顺序运行

**本章接口：** `coordinate_compress(values)`、`rank_transform(arr)`、`count_smaller_sorted_list(nums)`

### 导入本章使用的库

In [1]:
from bisect import bisect_left, insort

### coordinate_compress

In [2]:
def coordinate_compress(values):
    ordered = sorted(set(values))
    rank = {v: i for i, v in enumerate(ordered)}
    return (ordered, [rank[v] for v in values])

### rank_transform

In [3]:
def rank_transform(arr):
    ordered, ranks = coordinate_compress(arr)
    return [r + 1 for r in ranks]

### count_smaller_sorted_list

In [4]:
def count_smaller_sorted_list(nums):
    ordered = []
    answer = []
    for x in reversed(nums):
        answer.append(bisect_left(ordered, x))
        insort(ordered, x)
    return answer[::-1]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[10**12,-7,20,-7]; ordered,ranks=coordinate_compress(a)
show_table(['原值','零基秩'],list(zip(a,ranks)))
print('秩不能保持差值：',ordered[-1]-ordered[0],'≠',len(ordered)-1)

原值,零基秩
1000000000000,2
-7,0
20,1
-7,0


秩不能保持差值： 1000000000007 ≠ 2


## 7. 正确性、适用条件与复杂度

排序去重产生严格递增的一一映射，故比较关系保持不变；重复值共享同一秩。右向左处理时排序列表恰包含当前位置右侧所有值，二分左界排除了等值元素。

**代价：** 压缩O(n log n)时间O(n)空间；列表计数基线最坏O(n²)时间，因为insort插入要移动元素，二分O(log n)不代表整个操作O(log n)。返回(排序原值表,与输入对应的零基秩)。

### 展开理解

先说坐标压缩为什么正确。压缩后的有序表是严格递增的，而且每个原始值都对应表里唯一一个位置，所以"原来 a 比 b 小"这件事，在压缩后一定变成"a 的秩比 b 的秩小"；两个相等的值因为先去了重，必然共享同一个秩。也就是说，大小关系和相等关系都被原封不动地搬到了小整数上，只有"差多少"这个信息丢了——而这类题目本来就只需要相对次序。反过来说，如果你后面的算法需要用到真实差值（比如区间长度、距离和），坐标压缩就不能随便用。

再解释 count_smaller_sorted_list 里"用 bisect_left 是对的"：我们从右往左扫，处理到位置 i 的时候，列表 ordered 里装的恰好是原数组下标 i 右边的全部元素（一个不多一个不少，因为左边的还没走到）。`bisect_left` 找的是 x 应该插入的最左位置，列表里所有排在那个位置左边的元素都严格小于 x，排在该位置及右边的元素都大于等于 x，所以"插入位置 = 严格小于 x 的元素个数"，恰好是我们要的"右侧更小数个数"，并且正确地排除了和 x 相等的元素。

复杂度方面：`coordinate_compress` 里排序占大头，是 O(n log n) 时间、O(n) 空间。具体到数字，n = 10^5 时大概是一百多万次比较操作，一眨眼就算完。`count_smaller_sorted_list` 表面上每次查询只要 O(log n)，但 `insort` 插入要把后面的元素整体往后挪，最坏 O(n)，所以整体最坏是 O(n²)。n = 10^5 时最坏要一百亿次挪动，那就要等很久了——这正是 notebook 推导里说"后续再用 Fenwick 消除列表插入的线性代价"的原因：下一章的树状数组能把插入和查询都做到 O(log n)。这一章先用最容易写对的方式把"查询的含义"弄明白。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的测试分两部分：三条手工断言加上一个暴力对拍循环。

- `assert rank_transform([40,10,20,30])==[4,1,2,3]`：这是正常值测试。40 是最大的，秩为 4；10 最小，秩为 1；20 和 30 依次是 2 和 3。它验证排序和一基编号都对。
- `assert rank_transform([7,7,-2])==[2,2,1]`：这是重复值测试。两个 7 拿到同一个秩 2，说明去重生效、相等关系被保留；-2 最小拿 1，说明负数也没问题。
- `assert count_smaller_sorted_list([5,2,6,1])==[2,1,1,0]`：这是右侧更小数的正常用例，就是我们第三节手算过的那个例子，末尾的 1 是边界——最右边的元素右边没有数，结果必须是 0。
- 接下来的双重循环是暴力对拍：`for n in range(7)` 枚举所有长度 0 到 6 的序列，元素只能从 `(-100, 0, 10^12)` 里选——一个负数、一个零、一个超大数，专门覆盖"负数/零/巨值"三类特殊值，还包括空数组（n=0）和全相等数组。对每个序列它检查三件事：
  - `[ordered[r] for r in ranks]==list(a)`：拿秩去查有序表能还原出原数组，说明映射是一一对应的（这条同时验证了重复值共享秩：两个相同值还原回来还是相同值）。
  - `all((a[i]<a[j])==(ranks[i]<ranks[j]) ...)`：任何一对位置上，原值的大小关系和秩的大小关系完全一致，这就是"保序"的完整定义。
  - `count_smaller_sorted_list(a)==[sum(y<x for y in a[i+1:]) for i,x in enumerate(a)]`：把我们的实现和"对每个位置暴力数右边元素"的笨办法逐位对比，确保结果一致。

最后打印 `N134: 所有本章断言通过`，说明全部检查都过了。

这套对拍的设计值得学：它不是随机造数据，而是**穷举**短长度下的全部可能——长度 0 到 6、每个位置三选一，总共 1+3+9+...+729 = 1093 个序列。因为长度很短，穷举比随机更能保证"小输入上没有漏网之鱼"；而三元素值集刻意拉开了量级（-100 / 0 / 10^12），让"负数、零、天文数字、全相等、部分相等"这些边角情形全部被覆盖到。

如果只让你记住一句话：**坐标压缩 = 去重排序建字典，把"值"换成"名次"；它保留大小和相等关系，丢掉距离；右侧更小数 = 倒着扫 + 有序表现查**。后面树状数组、线段树的章节都会反复用到"先压缩、再当下标用"这个套路。

In [6]:
assert rank_transform([40, 10, 20, 30]) == [4, 1, 2, 3]

assert rank_transform([7, 7, -2]) == [2, 2, 1]

assert count_smaller_sorted_list([5, 2, 6, 1]) == [2, 1, 1, 0]

from itertools import product

for n in range(7):
    for a in product((-100, 0, 10 ** 12), repeat=n):
        ordered, ranks = coordinate_compress(a)
        assert [ordered[r] for r in ranks] == list(a)
        assert all(((a[i] < a[j]) == (ranks[i] < ranks[j]) for i in range(n) for j in range(n)))
        assert count_smaller_sorted_list(a) == [sum((y < x for y in a[i + 1:])) for i, x in enumerate(a)]

print('N134: 所有本章断言通过')

N134: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 构造压缩后距离改变的例子。

**练习 2：** 明确有序列表插入仍然线性。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1：构造压缩后距离改变的例子。** 提示：你只要找一组值，让"原始差值"和"秩的差值"对不上就行。比如取三个间距不等的数（一个间距特别大、一个特别小），压缩后相邻两对数的秩差都变成 1，但原始差值完全不同。写的时候先手算出输入、期望输出，再调用 `coordinate_compress` 对照。你甚至可以直接引用 “运行示例”部分 已经给出的例子：原值差 `10^12 - (-7)`，秩差却只有 2。验收时把"哪些信息被保留（大小、相等）、哪些被丢弃（距离）"说清楚，才算真正答到点上。另一个值得写进报告的边界：空数组 `[]` 进去，`ordered` 和 `ranks` 都是空的，不报错；单元素数组进去秩必是 0（一基是 1）。

**练习 2：明确有序列表插入仍然是线性。** 提示：重点解释 `insort` 的两段式——二分找位置是 O(log n)，但真正插入要把那个位置之后的所有元素往后挪一格，这段是 O(n)。你可以设计一个最坏情况来观察它：比如让每个新元素都要插在列表最前面（从右往左处理一个递增数组的场景，想想什么时候会发生），这时每次插入都要挪动整个列表。验证办法是对比不同 n 下的耗时增长趋势，看它是不是明显快于线性比例。一个现成的观察角度：处理 `[1, 2, 3, 4]`（递增数组）时我们从右往左先见到 4、再 3、再 2，每个新元素都比列表里所有元素小，全都插在最前面——每次都触发整表挪动。量级感受：n = 10^5 时最坏挪动约 100000²/2 = 五十亿次元素搬运，远慢于下一章树状数组的两百万次循环。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
from bisect import bisect_left, insort

def coordinate_compress(values):
    ordered = sorted(set(values))
    rank = {v: i for i, v in enumerate(ordered)}
    return (ordered, [rank[v] for v in values])

def rank_transform(arr):
    ordered, ranks = coordinate_compress(arr)
    return [r + 1 for r in ranks]

def count_smaller_sorted_list(nums):
    ordered = []
    answer = []
    for x in reversed(nums):
        answer.append(bisect_left(ordered, x))
        insort(ordered, x)
    return answer[::-1]

# 示例与回归测试
assert rank_transform([40, 10, 20, 30]) == [4, 1, 2, 3]

assert rank_transform([7, 7, -2]) == [2, 2, 1]

assert count_smaller_sorted_list([5, 2, 6, 1]) == [2, 1, 1, 0]

from itertools import product

for n in range(7):
    for a in product((-100, 0, 10 ** 12), repeat=n):
        ordered, ranks = coordinate_compress(a)
        assert [ordered[r] for r in ranks] == list(a)
        assert all(((a[i] < a[j]) == (ranks[i] < ranks[j]) for i in range(n) for j in range(n)))
        assert count_smaller_sorted_list(a) == [sum((y < x for y in a[i + 1:])) for i, x in enumerate(a)]

print('N134: 所有本章断言通过')

N134: 所有本章断言通过


## 11. 代表题与迁移

- **1331. Rank Transform of an Array（数组序号转换）**：这题就是 `rank_transform` 的原题——把每个元素换成它在全数组中的名次（从 1 开始，重复元素同名次），练的是"去重 + 排序 + 建值到秩的映射"这条完整链路。本题也直接暴露一基与零基的区别：题目要求最小元素拿到 1 而不是 0，所以封装里那步 `r+1` 不能省。
- **315. Count of Smaller Numbers After Self（右侧更小数）**：这题是 `count_smaller_sorted_list` 的原题，本章用"有序列表 + 二分"的基线写法把查询含义讲清楚，下一章会用树状数组把它优化到 O(n log n)（notebook 里标注为 preview，即预告性质）。

最后按 “掌握检查”部分 的自测标准检验自己：能不能不看代码说出"倒序扫描 + 先查后插"这个循环的不变式（处理位置 i 前，列表恰好装着 i 右边的所有元素）？能不能举出一个"压缩后距离改变"的数字例子？能不能解释为什么 `bisect_left` 而不是 `bisect_right`（前者排除等值元素，得到的是"严格更小"的个数）？三个都说得清楚，这一章就可以翻篇了。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。